In [1]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.preprocessing import StandardScaler, QuantileTransformer

In [2]:
# as the code is loaded from a subfolder, we use the following snippet to add meatcube2 to the import path
# for a normal usage with meatcube2 installed, the two following lines are unnecessary
import sys, os
try:
    current_folder = os.path.dirname(__file__) # normal way
except NameError:
    current_folder = globals()['_dh'][0] # jupyter notebook way
sys.path.append(os.path.join(current_folder, '../..'))

# we load meatcube2 
from meatcube2.models import MeATCubeCB, CtCoAT, CtCoATNaive, EnergyKNN
from meatcube2.plotting.compare import animate_dataset_model_grid_on_models, split_datasets, plot_dataset_model_grid
from meatcube2.cb_maintenance import CBClassificationMaintainer

In [6]:
from scipy.linalg import norm
from scipy.spatial.distance import euclidean, cosine, correlation, braycurtis, hamming
def euclidean_sim(x1,x2):
    return np.exp(-euclidean(x1,x2))
def class_equality_sim(y1,y2):
    return np.equal(y1,y2).astype(float)

In [34]:
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_circles, make_classification, make_moons, load_iris, load_breast_cancer, load_diabetes, load_wine
from sklearn.decomposition import PCA
from sklearn.base import ClassifierMixin


class InvertPCA(ClassifierMixin):
    def __init__(self, pca: PCA, clf: ClassifierMixin) -> None:
        super().__init__()
        self.clf = clf
        self.pca = pca
    def _transform(self, X):
        X_ = np.zeros((X.shape[0], self.pca.n_features_in_), dtype=X.dtype)
        X_[:,:X.shape[1]] = X
        X_ = self.pca.inverse_transform(X_)
        return X_
    def predict(self, X: np.ndarray, **kwargs):
        return self.clf.predict(self._transform(X), **kwargs)
    def predict_proba(self, X: np.ndarray, **kwargs):
        return self.clf.predict_proba(self._transform(X), **kwargs)
    def __sklearn_is_fitted__(self):
        return True
    def fit(self, X, y):
        pass

def plot_dataset(dataset, ax: plt.Axes, plot_ref=False, plot_test=False, alpha=0.7, pca=None):
    
    # prepare decision boundary kwargs
    cm = plt.cm.RdBu
    cm_bright = ListedColormap(["#FF0000", "#0033FF"])

    (X_train, y_train), (X_ref, y_ref), (X_test, y_test), = dataset
    X, y = np.concatenate([X_train, X_ref, X_test], axis=0), np.concatenate([y_train, y_ref, y_test], axis=0)

    if len(X[0]) > 2: # dimensionality reduction
        if pca is None:
            pca = PCA(n_components=len(X[0]), random_state=42)
        X_ = pca.fit_transform(X)
        dimensionality_reduction = True
        X_train_ =  pca.transform(X_train)[:,:2]
        X_ref_ =    pca.transform(X_ref)[:,:2]
        X_test_ =   pca.transform(X_test)[:,:2]
        X_ = X_[:,:2]
    else:
        dimensionality_reduction = False
        X_train_ = X_train
        X_ref_ = X_ref
        X_test_ = X_test
        X_ = X

    x_min, x_max = X_[:, 0].min() - 0.5, X_[:, 0].max() + 0.5
    y_min, y_max = X_[:, 1].min() - 0.5, X_[:, 1].max() + 0.5

    # Plot the ref points
    if X_ref is not None and y_ref is not None and plot_ref:
        ax.scatter(
            X_ref_[:, 0], X_ref_[:, 1], c=y_ref, cmap=cm_bright, alpha=alpha, #edgecolors="k",
                marker="^",
        #edgecolors="w",
                edgecolors="k",
                linewidths=0.25,
                label="Ref data"
        )

    # Plot the testing points
    if plot_test:
        scatter = ax.scatter(
            X_test_[:, 0],
            X_test_[:, 1],
            c=y_test,
            cmap=cm_bright,
            edgecolors="k",
            linewidths=0.25,
            alpha=alpha,
            marker="v",
            label="Test data"
        )

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(())
    ax.set_yticks(())
    return pca

def plot_model(dataset, classifier, classifier_name, ax: plt.Axes, size_as_decrement_score=False, pca=None):
    
    # prepare decision boundary kwargs
    DecisionBoundaryDisplay_kwargs = {"alpha":0.3, "grid_resolution":50}
    figure = ax.get_figure()
    cm = plt.cm.RdBu
    cm_bright = ListedColormap(["#FF0000", "#0033FF"])

    (X_train, y_train), (X_ref, y_ref), (X_test, y_test), = dataset
    X, y = np.concatenate([X_train, X_ref, X_test], axis=0), np.concatenate([y_train, y_ref, y_test], axis=0)


    if len(X[0]) > 2: # dimensionality reduction
        if pca is None:
            pca = PCA(n_components=len(X[0]), random_state=42)
        X_ = pca.fit_transform(X)
        dimensionality_reduction = True
        X_train_ =  pca.transform(X_train)[:,:2]
        X_ref_ =    pca.transform(X_ref)[:,:2]
        X_test_ =   pca.transform(X_test)[:,:2]
        X_ = X_[:,:2]
    else:
        dimensionality_reduction = False
        X_train_ = X_train
        X_ref_ = X_ref
        X_test_ = X_test
        X_ = X

    x_min, x_max = X_[:, 0].min() - 0.5, X_[:, 0].max() + 0.5
    y_min, y_max = X_[:, 1].min() - 0.5, X_[:, 1].max() + 0.5


    #clf_ = make_pipeline(scaler, clf)
    score = accuracy_score(classifier.predict(X_test), y_test)
    score = accuracy_score(classifier.predict(X_ref), y_ref)
    #classifier.score(X_test, y_test)

    if dimensionality_reduction:
        estimator = InvertPCA(pca, classifier)
    else: estimator = classifier
    DecisionBoundaryDisplay.from_estimator(
        estimator, X_, cmap=cm, ax=ax, eps=0.5, **DecisionBoundaryDisplay_kwargs
    )

    # Plot the CB
    _X = classifier._X
    _y = classifier._y
    if dimensionality_reduction:
        _X = pca.transform(_X)
    # if size_as_decrement_score:
    #     sizes = classifier.decrement_scores(X_ref, y_ref)
    #     s = ((sizes - sizes.min()) / (sizes.max() - sizes.min())) * 20 + 10
    #     s = np.nan_to_num(s, nan=30)
    #     scatter = ax.scatter(
    #         x=_X[:, 0],
    #         y=_X[:, 1],
    #         c=_y,
    #         cmap=cm_bright,
    #         edgecolors="k",
    #         alpha=1,
    #         marker="o",
    #         label="CB",
    #         s= s
    #         )
        
    # else:
    scatter = ax.scatter(
        x=_X[:, 0],
        y=_X[:, 1],
        c=_y,
        cmap=cm_bright,
        edgecolors="k",
        linewidths=0.5,
        alpha=.9,
        marker="o",
        label="CB"
    )

    # display additional information
    if isinstance(classifier, CBClassificationMaintainer):
        message_bonus = f" |CB|={classifier.initial_estimator_len_} -> {len(classifier)}"
    else:
        message_bonus = f" |CB|={len(classifier)}"
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(())
    ax.set_yticks(())
    ax.set_title(classifier_name + "\n" + (f"Acc. {score:.2%}") + message_bonus)#.lstrip("0")
    # ax.text(
    #     x_max - 0.3,
    #     y_min + 0.3,
    #     ("%.2f" % score).lstrip("0") + message_bonus,
    #     size=15,
    #     horizontalalignment="right",
    # )


def plot_dataset_model_legend(ax: plt.Axes):
    cm = plt.cm.RdBu
    cm_bright = ListedColormap(["#FF0000", "#0000FF"])
    # produce a legend with a cross-section of sizes from the scatter
    legend_elements = [
        plt.Line2D([0], [0], marker='o', color='black', label='CB', markersize=10, linestyle='None'),
        plt.Line2D([0], [0], marker='^', color='black', label='Ref', markersize=10, linestyle='None'),
        plt.Line2D([0], [0], marker='v', color='black', label='Test', markersize=10, linestyle='None'),
        plt.Line2D([0], [0], marker='o', color=cm_bright.colors[0], label='Class 1', markersize=10, linestyle='None'),
        plt.Line2D([0], [0], marker='o', color=cm_bright.colors[1], label='Class 2', markersize=10, linestyle='None'),
    ]
    
    ax.legend(handles=legend_elements, title="Data", loc='center left')

In [21]:
moon = make_moons(noise=0.2, random_state=2, n_samples=100)
dataset_split = split_datasets([moon,], True, ref_size=40, test_size=20)
dataset, dataset_name = moon, "Moons"

In [31]:
iris = load_iris(return_X_y=True)
dataset_split = split_datasets([iris,], True, ref_size=50, test_size=50)
dataset, dataset_name = iris, "Iris"

In [39]:
data = load_wine(return_X_y=True)
dataset_split = split_datasets([data,], True, ref_size=50, test_size=20)
dataset, dataset_name = data, "Wine"
dataset_split[0][0][1].shape

(108,)

In [41]:
data = load_breast_cancer(return_X_y=True)
dataset_split = split_datasets([data,], True, ref_size=50, test_size=20)
dataset, dataset_name = data, "Breast Cancer"
dataset_split[0][0][1].shape


(499,)

In [ ]:
(X_train, y_train), (X_ref, y_ref), (X_test, y_test) = dataset_split[0]
maintainer_coat = CBClassificationMaintainer(
    MeATCubeCB(euclidean_sim, class_equality_sim, precompute_sim_matrix=True),
    memorize_estimators=True,
    patience=-1)
maintainer_ctcoat = CBClassificationMaintainer(
    CtCoAT(euclidean_sim, class_equality_sim, precompute_sim_matrix=True),
    memorize_estimators=True,
    patience=-1)
maintainer_7nn = CBClassificationMaintainer(
    EnergyKNN(euclidean_sim, class_equality_sim, n_neighbors=7, precompute_sim_matrix=True),
    memorize_estimators=True,
    patience=-1)
maintainer_coat.fit(X_train, y_train, X_ref, y_ref)
maintainer_ctcoat.fit(X_train, y_train, X_ref, y_ref)
maintainer_7nn.fit(X_train, y_train, X_ref, y_ref)

from matplotlib.figure import Figure

SIZE_UNIT = 3
fig: Figure = plt.figure(figsize=(3*SIZE_UNIT, 3*SIZE_UNIT))
models_start = [maintainer_coat.estimators_[0], maintainer_ctcoat.estimators_[0], maintainer_7nn.estimators_[0]]
models_best = [maintainer_coat.best_estimator_, maintainer_ctcoat.best_estimator_, maintainer_7nn.best_estimator_]
for i, (model, model_name) in enumerate(zip(models_start, ["CoAT", "CtCoAT", "7NN"])):
    ax = fig.add_subplot(3,3,i+1)
    pca = plot_dataset(dataset_split[0], ax, False, False, alpha=0.6)
    pca = plot_model(dataset_split[0], model, model_name + " before $EC$", ax, pca = pca)
for i, (model, model_name) in enumerate(zip(models_best, ["CoAT", "CtCoAT", "7NN"])):
    ax = fig.add_subplot(3,3,3+i+1)
    plot_dataset(dataset_split[0], ax, False, False, alpha=0.6, pca = pca)
    plot_model(dataset_split[0], model, model_name + " after $EC$", ax, pca = pca)
cm_bright = ListedColormap(["#FF0000", "#0033FF"])

# produce a legend with a cross-section of sizes from the scatter
legend_elements = [
    plt.Line2D([0], [0], marker='o', color=cm_bright.colors[0], label='CB class 1', markersize=10, linestyle='None'),
    plt.Line2D([0], [0], marker='o', color=cm_bright.colors[1], label='CB class 2', markersize=10, linestyle='None'),
]

ax = fig.add_subplot(3,1,3)
ax.set_axis_off()
ax.legend(handles=legend_elements, #title="Data",
            #title="Data", loc='center left')
           loc='upper center', ncols=2)
maintainer_coat.best_index_, maintainer_ctcoat.best_index_, maintainer_7nn.best_index_